In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install spectral
import os, time, zipfile
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
import tensorflow as tf
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, accuracy_score, classification_report,
                             cohen_kappa_score, precision_recall_fscore_support)
from operator import truediv
from tensorflow.keras.utils import to_categorical
import spectral
import plotly.figure_factory as ff

# ============================================================
# CONFIG
# ============================================================
dataset = 'SA'
#test_ratio = 0.7
windowSize = 25
#samples_per_class = 15
K = 15
model_name = "SA_PCA_15_Light_perc_10_Optimized"
results_folder = "results"
os.makedirs(results_folder, exist_ok=True)

# ============================================================
# DATA LOADING
# ============================================================
def loadData(name):
    base = "/content/drive/MyDrive/Colab Notebooks/dataset/"
    if name == 'SA':
        #data = sio.loadmat("Salinas_corrected.mat")["salinas_corrected"]
        #labels   = sio.loadmat("Salinas_gt.mat")["salinas_gt"]
        data = sio.loadmat(base+'Salinas_corrected.mat')['salinas_corrected']
        labels = sio.loadmat(base+'Salinas_gt.mat')['salinas_gt']
    return data, labels

def applyPCA(X, numComponents):
    newX = np.reshape(X, (-1, X.shape[2]))
    pca = PCA(n_components=numComponents, whiten=True)
    newX = pca.fit_transform(newX)
    return np.reshape(newX, (X.shape[0], X.shape[1], numComponents)), pca

def padWithZeros(X, margin=2):
    newX = np.zeros((X.shape[0] + 2*margin, X.shape[1] + 2*margin, X.shape[2]))
    newX[margin:X.shape[0]+margin, margin:X.shape[1]+margin, :] = X
    return newX

def createImageCubes(X, y, windowSize=5):
    margin = windowSize//2
    zeroPaddedX = padWithZeros(X, margin=margin)
    patchesData, patchesLabels = [], []
    for r in range(margin, zeroPaddedX.shape[0]-margin):
        for c in range(margin, zeroPaddedX.shape[1]-margin):
            patch = zeroPaddedX[r-margin:r+margin+1, c-margin:c+margin+1]
            label = y[r-margin, c-margin]
            if label > 0:
                patchesData.append(patch)
                patchesLabels.append(label-1)
    return np.array(patchesData), np.array(patchesLabels)

'''def splitTrainTestSet(X, y, testRatio, samples_per_class):
    np.random.seed(345)
    train_idx, test_idx = [], []
    for cl in np.unique(y):
        idx = np.where(y==cl)[0]
        np.random.shuffle(idx)
        train_idx.extend(idx[:samples_per_class])
        test_idx.extend(idx[samples_per_class:])
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]'''

def splitTrainTestSet(X, y, train_ratio=None, samples_per_class=None, random_state=42):
    """
    Splits dataset into train and test sets by taking either:
    (a) a fixed percentage of samples from each class, OR
    (b) a fixed number of samples from each class.

    Args:
        X (np.ndarray): Feature matrix.
        y (np.ndarray): Labels.
        train_ratio (float): Fraction of samples per class to use for training.
        samples_per_class (int): Fixed number of samples per class to use for training.
        random_state (int): Random seed for reproducibility.

    Returns:
        Xtrain, Xtest, ytrain, ytest
    """
    np.random.seed(random_state)
    train_idx = []
    test_idx = []

    if train_ratio is None and samples_per_class is None:
        raise ValueError("Provide either train_ratio or samples_per_class.")

    for cl in np.unique(y):
        idx = np.where(y == cl)[0]
        np.random.shuffle(idx)

        if samples_per_class is not None:  # case (b) fixed samples
            n_train = min(samples_per_class, len(idx) - 1)  # keep at least 1 for test
        else:  # case (a) percentage
            n_train = max(1, int(len(idx) * train_ratio))

        train_idx.extend(idx[:n_train])
        test_idx.extend(idx[n_train:])

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

# ============================================================
# DATA PREP
# ============================================================
X, y = loadData(dataset)
X, pca = applyPCA(X, numComponents=K)
X, y = createImageCubes(X, y, windowSize=windowSize)

# ------------------ Usage Examples ------------------

# (a) Use fixed percentage (e.g., 30% training)
Xtrain, Xtest, ytrain, ytest = splitTrainTestSet(X, y, train_ratio=0.1)

# (b) Use fixed count (e.g., 15 samples per class)
#Xtrain, Xtest, ytrain, ytest = splitTrainTestSet(X, y, samples_per_class=15)

#Not required below line
#Xtrain, Xtest, ytrain, ytest = splitTrainTestSet(X, y, test_ratio, samples_per_class)


Xtrain = Xtrain.reshape(-1, windowSize, windowSize, K, 1)
Xtest  = Xtest.reshape(-1, windowSize, windowSize, K, 1)
ytrain = to_categorical(ytrain)
ytest  = to_categorical(ytest)

# ============================================================
# MODEL
# ============================================================
S, L, output_units = windowSize, K, 16
input_layer = tf.keras.layers.Input((S, S, L, 1))
x = tf.keras.layers.Conv3D(8, (3,3,7), activation='relu')(input_layer)
x = tf.keras.layers.Conv3D(16,(3,3,5), activation='relu')(x)
x = tf.keras.layers.Conv3D(32,(3,3,3), activation='relu')(x)
conv3d_shape = x.shape
x = tf.keras.layers.Reshape((conv3d_shape[1], conv3d_shape[2], conv3d_shape[3]*conv3d_shape[4]))(x)
x = tf.keras.layers.Conv2D(24, (3,3), activation='relu')(x)
x = tf.keras.layers.Conv2D(96, (3,3), activation='relu')(x)
x = tf.keras.layers.MaxPooling2D((2,2))(x)
x = tf.keras.layers.Conv2D(128,(3,3), activation='relu')(x)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.4)(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
x = tf.keras.layers.Dropout(0.4)(x)
output_layer = tf.keras.layers.Dense(output_units, activation='softmax')(x)

model = tf.keras.models.Model(input_layer, output_layer)
model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(0.001), metrics=['accuracy'])

# Save model summary
with open(os.path.join(results_folder, f"{model_name}_summary.txt"), "w") as f:
    model.summary(print_fn=lambda x: f.write(x + "\n"))

# ============================================================
# TRAIN
# ============================================================
tic = time.perf_counter()
history = model.fit(Xtrain, ytrain, batch_size=256, epochs=100, validation_data=(Xtest, ytest), verbose=2)
toc = time.perf_counter()
train_time = toc - tic

# ============================================================
# PLOTS
# ============================================================
plt.figure()
plt.plot(history.history['accuracy'], label="Train Acc")
plt.plot(history.history['val_accuracy'], label="Val Acc")
plt.plot(history.history['loss'], label="Train Loss")
plt.plot(history.history['val_loss'], label="Val Loss")
plt.xlabel("Epochs"); plt.ylabel("Value"); plt.legend()
plt.title("Training vs Validation")
plt.savefig(os.path.join(results_folder, f"{model_name}_training.png"))

# ============================================================
# EVALUATION
# ============================================================
tic1 = time.perf_counter()
y_pred = np.argmax(model.predict(Xtest), axis=1)
toc1 = time.perf_counter()
test_time = toc1 - tic1

true_y = np.argmax(ytest, axis=1)
classification = classification_report(true_y, y_pred, digits=4)
precision, recall, f1, _ = precision_recall_fscore_support(true_y, y_pred, average='weighted')
oa = accuracy_score(true_y, y_pred)
confusion = confusion_matrix(true_y, y_pred)
each_acc = np.nan_to_num(np.diag(confusion)/np.sum(confusion,axis=1))
aa = np.mean(each_acc)
kappa = cohen_kappa_score(true_y, y_pred)

# ============================================================
# SAVE RESULTS
# ============================================================
with open(os.path.join(results_folder, f"{model_name}_results.txt"), "w") as f:
    f.write(f"Training time: {train_time:.2f}s\n")
    f.write(f"Testing time: {test_time:.2f}s\n")
    f.write(f"Overall Accuracy: {oa*100:.2f}%\n")
    f.write(f"Average Accuracy: {aa*100:.2f}%\n")
    f.write(f"Kappa: {kappa*100:.2f}%\n")
    f.write(f"Precision: {precision*100:.2f}%\n")
    f.write(f"Recall: {recall*100:.2f}%\n")
    f.write(f"F1-score: {f1*100:.2f}%\n\n")
    f.write("Classwise accuracy:\n")
    f.write(str(each_acc*100)+"\n\n")
    f.write("Classification Report:\n")
    f.write(classification+"\n\n")
    f.write("Confusion Matrix:\n")
    f.write(str(confusion)+"\n")

# ============================================================
# PREDICT FULL MAP
# ============================================================
PATCH_SIZE = windowSize
X_full, y_full = loadData(dataset)
X_full, _ = applyPCA(X_full, numComponents=K)
X_pad = padWithZeros(X_full, PATCH_SIZE//2)

outputs = np.zeros(y_full.shape)
patches, coords = [], []
for i in range(y_full.shape[0]):
    for j in range(y_full.shape[1]):
        if y_full[i,j] > 0:
            patch = X_pad[i:i+PATCH_SIZE, j:j+PATCH_SIZE, :]
            patches.append(patch)
            coords.append((i,j))

patches = np.array(patches).reshape(-1, PATCH_SIZE, PATCH_SIZE, K, 1)
preds = np.argmax(model.predict(patches, batch_size=512), axis=1)

for (i,j), p in zip(coords, preds):
    outputs[i,j] = p+1

spectral.save_rgb(os.path.join(results_folder, f"{model_name}_classified_map.jpg"), outputs.astype(int), colors=spectral.spy_colors)
spectral.save_rgb(os.path.join(results_folder, f"{model_name}_groundtruth.jpg"), y_full, colors=spectral.spy_colors)

# ============================================================
# ZIP EVERYTHING
# ============================================================
zip_path = f"{model_name}_outputs.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for file in os.listdir(results_folder):
        zipf.write(os.path.join(results_folder, file), arcname=file)

print(f"✅ All outputs saved in {zip_path}")
